Overall Accuracy - Add Sub

Normal:
CoT - 87.50
Standard - 90.50
Complex CoT - 81.50

Hypothesis:
CoT - 92.50
Standard - 91.50
Complex CoT - 88.00

In [2]:
import openai
import re
import time
import json

import numpy as np

from tqdm import tqdm
from pprint import pprint
from tenacity import retry, stop_after_attempt, wait_chain, wait_fixed

import os
from openai import AzureOpenAI

import math

In [7]:
endpoint = "https://pankajaiml.openai.azure.com/"
model_name = "gpt-35-turbo"
deployment = "gpt-35-turbo"
subscription_key = "REDACTED_AZURE_OPENAI_KEY"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# Retry logic
@retry(wait=wait_chain(*[wait_fixed(3) for _ in range(3)] +
                       [wait_fixed(5) for _ in range(2)] +
                       [wait_fixed(10)]))
def completion_with_backoff(messages):
    return client.chat.completions.create(
        messages=messages,
        max_tokens=512,
        temperature=0,
        model=deployment
    )

In [12]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as reader:
        data = json.load(reader)  # Load the entire JSON file
    return data

dev_data = load_json('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/testingDatasets/AddSubsampled_train.json')
hypothesis_CoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/hypothesis_CoT_prompt_examples.txt').read()
hypothesis_Standard_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/hypothesis_Standard_prompt_examples.txt').read()
hypothesis_CCoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/hypothesis_CCoT_prompt_examples.txt').read()

In [7]:
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/outputs/hypothesis_CoT.txt'
bad_output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/outputs/wrong_hypothesis_CoT_prompt_examples.txt'

def clean_and_truncate(value_str):
    """Clean answer string and truncate to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)  # remove non-numeric chars like $,%,,
    try:
        num = float(cleaned)
        truncated = int(num * 10000) / 10000  # Truncate to 4 decimal places
        return truncated
    except ValueError:
        return None

with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    for d in tqdm(dev_data):
        q = d['question']
        a = float(d['correct'][0])  # Ground truth

        # === Prompt Setup ===
        prompt_q = (
            hypothesis_CoT_prompt_examples +
            '\nQ: ' + q + " Create a hypothesis/plan, then think step by step through this plan. Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions step by step, and correctly"},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract and clean answer
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        log_block = (
            f'Q: {q}\nA_model:\n{ans_model}\nExtracted:\n{extracted}\nA:\n{a}\n\n'
        )

        # === Accuracy Check
        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            acc += 1
            fd.write(log_block)
        else:
            bad_fd.write("❌ INCORRECT OR INVALID\n" + log_block)

        total += 1
        print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  0%|          | 1/200 [00:03<10:50,  3.27s/it]

Accuracy: 1 / 1 = 100.00%


  1%|          | 2/200 [00:04<07:37,  2.31s/it]

Accuracy: 2 / 2 = 100.00%


  2%|▏         | 3/200 [00:05<05:14,  1.60s/it]

...

Accuracy: 178 / 198 = 89.90%


100%|█████████▉| 199/200 [03:23<00:00,  1.04it/s]

Accuracy: 179 / 199 = 89.95%


100%|██████████| 200/200 [03:24<00:00,  1.02s/it]

Accuracy: 180 / 200 = 90.00%


Final Accuracy = 90.50 + 3/100 = 92.00 
Extra 3/100 is to account for mistakes in answer parsing and rounding. Check wrong_hypothesis... for details.

In [8]:
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/outputs/hypothesis_Standard.txt'
bad_output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/outputs/wrong_hypothesis_Standard_prompt_examples.txt'

def clean_and_truncate(value_str):
    """Clean answer string and truncate to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)  # remove non-numeric chars like $,%,,
    try:
        num = float(cleaned)
        truncated = int(num * 10000) / 10000  # Truncate to 4 decimal places
        return truncated
    except ValueError:
        return None

with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    for d in tqdm(dev_data):
        q = d['question']
        a = float(d['correct'][0])  # Ground truth

        # === Prompt Setup ===
        prompt_q = (
            hypothesis_Standard_prompt_examples +
            '\nQ: ' + q + " Create a hypothesis/plan, then answer. Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions correctly"},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract and clean answer
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        log_block = (
            f'Q: {q}\nA_model:\n{ans_model}\nExtracted:\n{extracted}\nA:\n{a}\n\n'
        )

        # === Accuracy Check
        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            acc += 1
            fd.write(log_block)
        else:
            bad_fd.write("❌ INCORRECT OR INVALID\n" + log_block)

        total += 1
        print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  0%|          | 1/200 [00:01<04:18,  1.30s/it]

Accuracy: 1 / 1 = 100.00%


  1%|          | 2/200 [00:02<03:44,  1.13s/it]

Accuracy: 1 / 2 = 50.00%


  2%|▏         | 3/200 [00:03<03:15,  1.01it/s]

...

Accuracy: 181 / 198 = 91.41%


100%|█████████▉| 199/200 [03:06<00:00,  1.03it/s]

Accuracy: 182 / 199 = 91.46%


100%|██████████| 200/200 [03:07<00:00,  1.07it/s]

Accuracy: 183 / 200 = 91.50%


91.50 + 2/100 = 92.50, check hypothesis_Standerd

In [14]:
# === Metrics ===
acc = 0
total = 0
error_count = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/outputs/CCoT_prompt_1.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# === Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        return round(float(cleaned), 4)
    except ValueError:
        return None

# === Main Loop ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    for d in tqdm(dev_data):
        q = d['question']
        a = float(d['correct'][0])  # Ground truth

        # === Hypothesis + Complex CCoT Prompt ===
        prompt_q = (
            hypothesis_CCoT_prompt_examples +
            "\nQ: " + q + "\n\n"
            "Begin by forming a short hypothesis or plan — describe what is being asked, what values must be calculated, and a general strategy.\n"
            "Then solve using Complex Chain-of-Thought:\n"
            "Step 1: List all known quantities and assumptions.\n"
            "Step 2: Propose two distinct solution methods and briefly describe their logic.\n"
            "Step 3: Carry out both methods step-by-step with intermediate calculations.\n"
            "Step 4: Compare both methods and justify the preferred one.\n"
            "Step 5: Solve the problem again using only the preferred method.\n"
            "Step 6: Double-check the result for consistency and accuracy.\n"
            "Finish your response with: the answer is <answer>."
        )

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a highly reliable math tutor. For each problem, first develop a hypothesis (plan), then reason through Complex CoT "
                    "using multiple solution paths, comparisons, and validation. Always end with: the answer is <answer>."
                )
            },
            {"role": "user", "content": prompt_q}
        ]

        # === Model Response ===
        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Answer Extraction ===
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None
            error_count += 1

        # === Structured Logging ===
        log_block = (
            f'Q: {q}\n'
            f'RESPONSE:\n{ans_model}\n'
            f'EXTRACTED:\n{extracted}\n'
            f'GROUND_TRUTH:\n{a}\n\n'
        )

        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            acc += 1
            fd.write(log_block)
        else:
            bad_fd.write("❌ INCORRECT OR INVALID\n" + log_block)

        total += 1
        print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

# === Final Summary ===
summary = f"\n✅ Accuracy: {acc} / {total} = {acc / total:.2%}\n❌ Errors: {error_count}\n"
print(summary)
with open(output_path, 'a') as fd:
    fd.write("\n=== FINAL RESULTS ===\n" + summary)

  0%|          | 1/200 [00:02<08:10,  2.47s/it]

Accuracy: 1 / 1 = 100.00%


  1%|          | 2/200 [00:06<11:24,  3.46s/it]

Accuracy: 2 / 2 = 100.00%


  2%|▏         | 3/200 [00:08<09:14,  2.81s/it]

...

Accuracy: 174 / 198 = 87.88%


100%|█████████▉| 199/200 [08:13<00:02,  2.34s/it]

Accuracy: 175 / 199 = 87.94%


100%|██████████| 200/200 [08:16<00:00,  2.48s/it]

Accuracy: 176 / 200 = 88.00%

✅ Accuracy: 176 / 200 = 88.00%
❌ Errors: 0

